# Can we predict `passed_quiz`?
Protocol: `protocol.yaml`. Holdout rows from `holdout.csv` (group holdout by `student_key`), 5-fold GroupKFold CV on training rows, model chosen by mean CV ROC AUC, holdout scored once.

**Feature timing (from columns.csv).** The quiz closes the session, so `session_minutes` (length of the whole session) is only final after the quiz ends. It is not known at prediction time and is excluded. `hints_used` is counted during the session, before the closing quiz, so it is kept (assumption: prediction is made just before the quiz). `prior_score` (placement test), `enrol_year`, `school_code` are known beforehand. `student_key` is an entity key and is excluded.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data.csv')
hold_rows = pd.read_csv('holdout.csv')['row'].to_numpy()
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
is_hold = np.zeros(len(df), bool); is_hold[hold_rows] = True
train, hold = df[~is_hold], df[is_hold]
print(len(train), len(hold), set(train.student_key) & set(hold.student_key))
TARGET='passed_quiz'
FEATURES=['enrol_year','school_code','prior_score','hints_used']
print(train[TARGET].mean(), hold[TARGET].mean())

## Candidates (named before any scoring)

In [ ]:
candidates = {
    'baseline_prior': DummyClassifier(strategy='prior'),
    'logreg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'logreg_C0.1': make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000)),
    'random_forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1),
    'hist_gb': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}
print(list(candidates))

## Cross-validation on training rows (GroupKFold by student; preprocessing is inside each pipeline)

In [ ]:
X, y, g = train[FEATURES], train[TARGET].to_numpy(), train['student_key']
folds = list(GroupKFold(n_splits=5).split(X, y, g))
cv_scores = {}
for name, m in candidates.items():
    s=[]
    for tr, va in folds:
        mm = clone(m).fit(X.iloc[tr], y[tr])
        s.append(float(roc_auc_score(y[va], mm.predict_proba(X.iloc[va])[:,1])))
    cv_scores[name]=s
cv = pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
print(cv.sort_values('mean', ascending=False))
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen:', chosen)

## Refit chosen candidate on all training rows; score holdout once

In [ ]:
final = clone(candidates[chosen]).fit(X, y)
holdout_auc = float(roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:,1]))
print('holdout roc_auc', holdout_auc)

In [ ]:
train_idx = np.flatnonzero(~is_hold)
evidence = {
  'features': FEATURES,
  'candidates': list(candidates),
  'cv_folds': [[train_idx[tr].tolist(), train_idx[va].tolist()] for tr, va in folds],
  'cv_scores': cv_scores,
  'chosen': chosen,
  'holdout_rows': hold_rows.tolist(),
  'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))